# AI Finance Assistant - model training notebook
Trains and evaluates the two ML models from the SRS (section 11): **expense classification** and **spending forecast**. The same logic lives in `ml/train.py`; this notebook is the step-by-step, explained version.

Run from the `notebooks/` folder, or from the project root after `cd ..`.

In [1]:
import sys, os
os.chdir(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from ml.train import load_and_clean, build_pipeline, CATEGORIES
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, r2_score

## 1. Load and clean the data (SRS 14)
Steps: drop duplicate `transaction_id`s, drop rows with missing description/amount/category/date, normalise category casing, keep positive amounts, parse dates.
The data is **synthetic** (`ml/generate_dataset.py`) - no real banking data is used.

In [2]:
raw = pd.read_csv('data/transactions.csv')
print('raw shape:', raw.shape)
print(raw.isna().sum())
df, log = load_and_clean()
print(log)
df.head()

raw shape: (3690, 7)
transaction_id        0
date                  0
description          25
amount                0
transaction_type      0
category              0
merchant            857
dtype: int64
{'raw_rows': 3690, 'after_dedup': 3660, 'after_dropping_missing': 3635, 'clean_rows': 3635}


,transaction_id,date,description,amount,transaction_type,category,merchant
0,1,2025-07-09,vegetables and fruits,253.63,expense,Food,Pizza Hut
1,2,2026-08-16,Swiggy ice cream,162.38,expense,Food,Swiggy
2,3,2025-10-30,BigBasket snacks,133.68,expense,Food,BigBasket
3,4,2026-06-29,bought tshirt pantaloons,2982.69,expense,Shopping,Pantaloons
4,5,2025-05-01,online payment,91.61,expense,Entertainment,NaN


In [3]:
df[df.transaction_type=='expense'].category.value_counts().plot(kind='bar', title='Expense class balance'); plt.show()

## 2. Expense classifier (SRS 11.1)
Features: TF-IDF on the cleaned description (word 1-2 grams + character 2-5 grams, so typos like *strbucks* still work). Target: category.

In [4]:
exp = df[(df.transaction_type=='expense') & df.category.isin(CATEGORIES)]
X_tr, X_te, y_tr, y_te = train_test_split(exp.description, exp.category, test_size=0.2, random_state=42, stratify=exp.category)
for name, clf in {'Logistic Regression': LogisticRegression(max_iter=2000, C=10), 'Naive Bayes': MultinomialNB(alpha=0.1)}.items():
    pipe = build_pipeline(clf).fit(X_tr, y_tr)
    print(name, 'accuracy:', round(pipe.score(X_te, y_te), 4))

Logistic Regression accuracy: 0.8516
Naive Bayes accuracy: 0.8501


In [5]:
model = build_pipeline(LogisticRegression(max_iter=2000, C=10)).fit(X_tr, y_tr)
pred = model.predict(X_te)
print(classification_report(y_te, pred, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred, labels=CATEGORIES), display_labels=CATEGORIES).plot(xticks_rotation=40, cmap='Blues'); plt.show()

               precision    recall  f1-score   support

        Bills       0.78      0.86      0.82        84
    Education       0.96      0.85      0.90        84
Entertainment       0.89      0.84      0.86        83
         Food       0.88      0.78      0.83        83
       Health       0.76      0.86      0.81        83
        Other       0.82      0.87      0.84        83
     Shopping       0.89      0.90      0.90        84
    Transport       0.87      0.86      0.86        83

     accuracy                           0.85       667
    macro avg       0.86      0.85      0.85       667
 weighted avg       0.86      0.85      0.85       667



In [6]:
for text in ['swiggy dinner 450', 'uber ride 300', 'netflix subscription', 'strbucks coffee', 'upi payment to ravi']:
    probs = model.predict_proba([text])[0]
    print(f'{text:25s} -> {model.classes_[probs.argmax()]:14s} ({probs.max():.0%})')

swiggy dinner 450         -> Food           (98%)
uber ride 300             -> Transport      (98%)
netflix subscription      -> Entertainment  (96%)
strbucks coffee           -> Food           (96%)
upi payment to ravi       -> Shopping       (44%)


Accuracy is ~85%, not 100%, on purpose: the dataset contains ambiguous rows (generic UPI/payment text, shops that sell everything). Those are genuinely unpredictable from the description alone, as in real life.

## 3. Spending forecast (SRS 11.2)
Linear Regression on monthly expense totals, chronological train/test split, evaluated with MAE / MSE / RMSE / R² and compared against a naive mean baseline.

In [7]:
e = df[df.transaction_type=='expense']
monthly = e.groupby(e.date.dt.to_period('M')).amount.sum().iloc[:-1]  # last month may be partial
y = monthly.values; n_test = max(3, len(y)//5)
X = np.arange(len(y)).reshape(-1,1)
lr = LinearRegression().fit(X[:-n_test], y[:-n_test]); p = lr.predict(X[-n_test:])
base = np.full(n_test, y[:-n_test].mean())
print('LinearRegression  MAE %.0f  RMSE %.0f  R2 %.2f' % (mean_absolute_error(y[-n_test:], p), mean_squared_error(y[-n_test:], p)**.5, r2_score(y[-n_test:], p)))
print('Mean baseline     MAE %.0f  RMSE %.0f' % (mean_absolute_error(y[-n_test:], base), mean_squared_error(y[-n_test:], base)**.5))

LinearRegression  MAE 37585  RMSE 43152  R2 -3.14
Mean baseline     MAE 37991  RMSE 43509


The synthetic dataset has no real spending trend (dates are random), so Linear Regression cannot beat the simple average here. That is an honest result: with real user data that grows or shrinks over time the trend term helps, and the app falls back to a plain average when a user has fewer than 3 months of history.

## 4. Save
Run `python -m ml.train` to train on all data and write `models/category_model.joblib` and `models/metrics.json`.